# Lab 10 — Build and Connect an MCP Server

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-10-build-and-connect-an-mcp-server/lab-10-build-and-connect-an-mcp-server.ipynb)

**Topic:** 5 — MCP and Sub-Agents

**Objective:** Orchestrate tools through the Model Context Protocol

Write your own MCP server exposing typed tools, then connect it to a coding agent. Because MCP is an open standard, the same server works with any MCP-aware client — write the capability once, use it everywhere.

Full step-by-step instructions are in the Learner Guide.


> **Version note.** This lab uses the `FastMCP` class from the `mcp` 1.x line. In `mcp` 2.0.0 the class was renamed to `MCPServer` and `FastMCP` was removed, so the install below pins `mcp[cli]>=1.9,<2`. On 2.0.0 the code is identical apart from `from mcp.server.mcpserver import MCPServer` and `MCPServer("toolbox")`.

The `[cli]` extra installs the `mcp` command-line tool, which includes the inspector.


**No API key cell in this lab.** The MCP server needs no model API key: only `get_weather` reaches the network, and Open-Meteo is key-free. Labs that do call a model API set their keys from Colab Secrets (`from google.colab import userdata`) with a getpass fallback — never a hard-coded key.


In [ ]:
!pip install -q "mcp[cli]>=1.9,<2" requests


No API key is needed for this lab: only `get_weather` reaches the network, and it calls the key-free Open-Meteo API. Registering the server with Claude Code (steps 5-7) is done on your own machine.


## 1. The local data file the server reads


In [ ]:
%%writefile team_notes.json
{
  "onboarding": "New joiners get a laptop on day 1 and complete security training in week 1.",
  "deployment": "Deploys run Tuesday and Thursday at 14:00 SGT. Fridays are frozen.",
  "oncall": "The on-call rotation changes Monday 09:00. Escalate to the lead after 30 minutes."
}


## 2. Write the server with three typed tools

The `@mcp.tool()` decorator reads the type hints and docstring to build the schema the client sees — the same principle as `@function_tool` in Lab 05, but exposed over a protocol rather than inside one process. Write the docstrings carefully: they are the only thing the agent reads when deciding whether a tool is relevant.

Because the protocol occupies stdout, **never use `print()` in an MCP stdio server**. A stray print corrupts the JSON-RPC stream and the client will fail to connect. Log to stderr or a file instead.

Every tool returns a descriptive error string rather than raising: a returned error is information the agent can act on, whereas a raised exception is a protocol-level failure the agent cannot reason about.


In [ ]:
%%writefile server.py
"""An MCP server exposing three typed tools over stdio."""

import json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("toolbox")

DATA_FILE = Path(__file__).parent / "team_notes.json"


@mcp.tool()
def current_time(timezone: str = "Asia/Singapore") -> str:
    """Get the current date and time in a given IANA timezone.

    Use this whenever the user asks what time it is, or needs today's date.

    Args:
        timezone: An IANA timezone name, e.g. "Asia/Singapore" or "Europe/London".
    """
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        return f"Error: '{timezone}' is not a valid IANA timezone name."
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")


@mcp.tool()
def search_team_notes(query: str) -> str:
    """Search the internal team handbook for a policy or process.

    Use this for questions about internal team practice such as onboarding,
    deployment schedules or the on-call rotation.

    Args:
        query: Keywords to search for, e.g. "deployment schedule".
    """
    if not DATA_FILE.exists():
        return "Error: the team notes file is missing."

    try:
        notes = json.loads(DATA_FILE.read_text())
    except json.JSONDecodeError as exc:
        return f"Error: the team notes file is malformed ({exc})."

    terms = query.lower().split()
    hits = [
        f"{topic}: {text}"
        for topic, text in notes.items()
        if any(term in topic.lower() or term in text.lower() for term in terms)
    ]
    return "\n".join(hits) if hits else f"No notes matched '{query}'."


@mcp.tool()
def get_weather(latitude: float, longitude: float) -> str:
    """Get the current temperature and wind speed for a location.

    Use this whenever the user asks about current weather conditions.

    Args:
        latitude: Latitude in decimal degrees, e.g. 1.29 for Singapore.
        longitude: Longitude in decimal degrees, e.g. 103.85 for Singapore.
    """
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m",
            },
            timeout=10,
        )
        response.raise_for_status()
        current = response.json()["current"]
    except requests.RequestException as exc:
        return f"Error: weather service unavailable ({exc})."
    except (KeyError, ValueError) as exc:
        return f"Error: unexpected response from weather service ({exc})."

    return (
        f"Temperature {current['temperature_2m']}degC, "
        f"wind {current['wind_speed_10m']} km/h."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")


## 3. Verify the server with a real MCP client

This is the same discovery-and-invoke handshake Claude Code performs when you run `claude mcp add`; doing it in code makes the protocol visible and lets you verify the server without an agent.

A stdio server waits silently for JSON-RPC on standard input — running `python server.py` directly and seeing no output means it started correctly.


In [ ]:
%%writefile client_demo.py
"""Connect to server.py over stdio, list its tools, and call each one."""

import asyncio
from pathlib import Path
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = Path(__file__).parent / "server.py"


def render(result) -> str:
    """Flatten an MCP tool result into plain text for printing."""
    return "\n".join(
        block.text for block in result.content if getattr(block, "type", "") == "text"
    )


async def demo() -> None:
    """Spawn the server, list its tools, then call each one."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("=== Tools discovered ===")
            for tool in tools.tools:
                first_line = (tool.description or "").strip().splitlines()[:1]
                print(f"  {tool.name}: {first_line[0] if first_line else ''}")

            print("\n=== current_time ===")
            result = await session.call_tool(
                "current_time", {"timezone": "Asia/Singapore"}
            )
            print(render(result))

            print("\n=== search_team_notes ===")
            result = await session.call_tool(
                "search_team_notes", {"query": "deployment schedule"}
            )
            print(render(result))

            print("\n=== get_weather (Singapore) ===")
            result = await session.call_tool(
                "get_weather", {"latitude": 1.29, "longitude": 103.85}
            )
            print(render(result))

            # The failure path: a returned error is information the agent can
            # act on, not a protocol-level crash.
            print("\n=== current_time with a bad timezone ===")
            result = await session.call_tool(
                "current_time", {"timezone": "Mars/Olympus_Mons"}
            )
            print(render(result))


def main() -> None:
    """Entry point."""
    asyncio.run(demo())


if __name__ == "__main__":
    main()


## 4. Run the client demo

All three tools should be discovered and return sensible results, and the bad timezone should produce the descriptive error rather than a traceback.


In [ ]:
!python client_demo.py


## 5. Register the server with your MCP client

This part is done on your own machine, not in Colab. Use an **absolute** path — the client does not resolve relative paths from your shell's working directory (get it with `pwd`):

```bash
claude mcp add my-tools -- python /absolute/path/to/server.py
```

If you installed into a virtual environment, point at that environment's interpreter so the `mcp` package is importable:

```bash
claude mcp add my-tools -- /absolute/path/to/lab-10-mcp/.venv/bin/python /absolute/path/to/lab-10-mcp/server.py
```

Then confirm discovery:

```bash
claude mcp list
```

Inside a `claude` session, `/mcp` shows the servers and their tools — you should see all three. If the server shows as failed, the usual causes are a relative path, the wrong Python interpreter, a `print()` in the server, or an import error. Run `python server.py` directly to surface a traceback.

The inspector is the fastest debugging loop before involving an agent:

```bash
mcp dev server.py
```


## 6. Ask the agent a question that requires your tools

Start `claude` in any folder and ask something the model cannot answer from its own knowledge:

```
When is our next deployment window, and what time is it now in Singapore?
```

The agent should call `search_team_notes` and `current_time`, then combine both results. Try a multi-tool question that requires sequencing:

```
What is the weather in Singapore right now (1.29, 103.85), and does that
affect our deployment schedule?
```

This is the payoff of MCP: you wrote these tools once, and any MCP-aware client — Claude Code, Claude Desktop, or an agent you build yourself — can use them without a bespoke integration.
